## Crossover sensitivity

**Calculate percentage of patients that are high risk at different crossover thresholds: 14d, 30d, and 60dd**

In [1]:
import numpy as np
import pandas as pd

## Import data

In [2]:
treatment_df = pd.read_csv('../outputs/pembro_chemo_index.csv')

In [3]:
treatment_df.sample(3)

,PatientID,LineName,StartDate
19869,FBFB024F5225B,chemo,2024-07-16 00:00:00
580,FA4A217CBABD5,pembro,2024-08-14
23445,FE32ABCE2AB96,chemo,2022-05-17 00:00:00


In [4]:
treatment_df.shape

(26405, 3)

In [5]:
treatment_df['treatment'] = (treatment_df['LineName'] == 'pembro').astype(int)

In [6]:
dtype_map = pd.read_csv('../outputs/pembro_chemo_features_dtypes.csv', index_col = 0).iloc[:, 0].to_dict()
features_df = pd.read_csv('../outputs/pembro_chemo_features_df.csv', dtype = dtype_map)

In [7]:
features_df.shape

(1208, 170)

In [8]:
df = pd.merge(features_df, treatment_df, on = 'PatientID', how = 'left')

In [9]:
df.shape

(1208, 173)

In [10]:
surv_pred_df = pd.read_csv('../outputs/gb_6m_survival_predictions_calibrated.csv')

In [11]:
surv_pred_df.shape

(1043, 2)

In [12]:
df = pd.merge(df, surv_pred_df, on = 'PatientID', how = 'left')

In [13]:
df.shape

(1208, 174)

In [14]:
df['StartDate'] = df['StartDate'].astype(str).str.split(' ').str[0]
df['StartDate'] = pd.to_datetime(df['StartDate'])

In [15]:
df['treatment_year'] = df['StartDate'].dt.year

In [16]:
df = df.query('treatment_year <= 2023')

In [17]:
df.shape

(1043, 175)

In [18]:
with open('../outputs/crossover_survival_estimate.txt', 'r') as f:
    crossover_survival_estimate_30 = float(f.read())

with open('../outputs/crossover_survival_estimate_14.txt', 'r') as f:
    crossover_survival_estimate_14 = float(f.read())

with open('../outputs/crossover_survival_estimate_60.txt', 'r') as f:
    crossover_survival_estimate_60 = float(f.read())

In [19]:
print(f'r* for 14d crossover: {crossover_survival_estimate_14}')
print(f'r* for 30d crossover: {crossover_survival_estimate_30}')
print(f'r* for 60d crossover: {crossover_survival_estimate_60}')

r* for 14d crossover: 1.8920023342158727
r* for 30d crossover: 1.676399262104699
r* for 60d crossover: 1.2721435018962486


In [20]:
print(f'percent high risk at 14d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_14').shape[0]/df.shape[0]}')
print(f'percent high risk at 30d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_30').shape[0]/df.shape[0]}')
print(f'percent high risk at 60d crossover: {df.query('psurv_180_calibrated < @crossover_survival_estimate_60').shape[0]/df.shape[0]}')

percent high risk at 14d crossover: 1.0
percent high risk at 30d crossover: 1.0
percent high risk at 60d crossover: 1.0
